In [1]:
import os
import re
import json
import time
import random
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd

import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

from sklearn.metrics import classification_report, f1_score, confusion_matrix


In [2]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)


DEVICE: cpu


In [3]:
# Resolve paths against the repository root, so this notebook works whether
# it is launched from notebooks/ or from the repository root.
ROOT = ".." if os.path.basename(os.getcwd()) == "notebooks" else "."
DATA_DIR = os.path.join(ROOT, "data", "processed")
TRAIN_PATH = os.path.join(DATA_DIR, "train.csv")
DEV_PATH   = os.path.join(DATA_DIR, "dev.csv")
TEST_PATH  = os.path.join(DATA_DIR, "test.csv")

train_df = pd.read_csv(TRAIN_PATH)
dev_df   = pd.read_csv(DEV_PATH)
test_df  = pd.read_csv(TEST_PATH)

print("Shapes:", {"train": train_df.shape, "dev": dev_df.shape, "test": test_df.shape})
print("Train columns:", train_df.columns.tolist())


Shapes: {'train': (3569, 22), 'dev': (950, 22), 'test': (1049, 4)}
Train columns: ['tweet_id', 'label', 'event', 'source_id', 'source_text', 'reply_text', 'reply_text_empty', 'source_text_empty', 'reply_text_norm', 'source_text_norm', 'reply_tokens', 'source_tokens', 'reply_len_tok', 'source_len_tok', 'reply_len_char', 'source_len_char', 'stance', 'overlap_ratio', 'jaccard', 'src_covered', 'reply_word_types', 'source_word_types']


In [4]:
def pick_first_existing(df: pd.DataFrame, candidates: List[str]) -> Optional[str]:
    for c in candidates:
        if c in df.columns:
            return c
    return None

SOURCE_CANDS = ["source_text_norm", "source_text", "source", "src_text", "source_tweet", "src"]
REPLY_CANDS  = ["reply_text_norm", "reply_text", "reply", "response", "reply_tweet", "tgt"]
LABEL_CANDS  = ["label", "stance", "gold", "y"]

def unify_split_columns(df: pd.DataFrame, split_name: str) -> pd.DataFrame:
    df = df.copy()
    src = pick_first_existing(df, SOURCE_CANDS)
    rpl = pick_first_existing(df, REPLY_CANDS)
    lbl = pick_first_existing(df, LABEL_CANDS)

    if src is None or rpl is None:
        raise ValueError(f"[{split_name}] Missing source/reply cols. Columns={df.columns.tolist()}")

    df["source_text__unified"] = df[src].astype(str)
    df["reply_text__unified"]  = df[rpl].astype(str)
    df["label_raw__unified"]   = df[lbl] if lbl is not None else np.nan

    print(f"[{split_name}] src='{src}' rpl='{rpl}' lbl='{lbl}'")
    return df

train_df = unify_split_columns(train_df, "train")
dev_df   = unify_split_columns(dev_df, "dev")
test_df  = unify_split_columns(test_df, "test")

SRC_COL = "source_text__unified"
RPL_COL = "reply_text__unified"
LBL_COL = "label_raw__unified"

print("Unified columns ready.")


[train] src='source_text_norm' rpl='reply_text_norm' lbl='label'
[dev] src='source_text_norm' rpl='reply_text_norm' lbl='label'
[test] src='source_text' rpl='reply_text' lbl='label'
Unified columns ready.


In [5]:
LABELS_CANON = ["support", "deny", "query", "comment"]
CANON_TO_PRETTY = {"support":"Support", "deny":"Deny", "query":"Query", "comment":"Comment"}

CODE_TO_LABEL = {"A": "support", "B": "deny", "C": "query", "D": "comment"}
LABEL_TO_CODE = {v: k for k, v in CODE_TO_LABEL.items()}
CODE_WORDS = ["A", "B", "C", "D"]

def normalise_gold_label(x) -> str:
    if pd.isna(x):
        return "comment"
    s = str(x).strip().lower()
    if s in LABELS_CANON:
        return s
    if s in ["s", "supporting", "supported"]:
        return "support"
    if s in ["d", "denying", "denial", "refute", "refuting"]:
        return "deny"
    if s in ["q", "querying", "question", "asking"]:
        return "query"
    if s in ["c", "commenting", "none"]:
        return "comment"
    # fallback
    if "support" in s: return "support"
    if "deny" in s or "refut" in s or "false" in s: return "deny"
    if "query" in s or "question" in s or "ask" in s: return "query"
    if "comment" in s: return "comment"
    return "comment"

for df in [train_df, dev_df, test_df]:
    df["gold"] = df[LBL_COL].apply(normalise_gold_label)

print("Train gold counts:\n", train_df["gold"].value_counts())
print("Dev gold counts:\n", dev_df["gold"].value_counts())
print("Test gold counts:\n", test_df["gold"].value_counts())


Train gold counts:
 gold
comment    2291
support     743
deny        273
query       262
Name: count, dtype: int64
Dev gold counts:
 gold
comment    616
support    167
query       96
deny        71
Name: count, dtype: int64
Test gold counts:
 gold
comment    778
query      106
support     94
deny        71
Name: count, dtype: int64


In [6]:
MODEL_NAME = "google/flan-t5-large"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

use_fp16 = (DEVICE == "cuda")
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if use_fp16 else torch.float32
).to(DEVICE)

model.eval()
print("Loaded:", MODEL_NAME, "| fp16:", use_fp16, "| device:", DEVICE)

tokenizer.truncation_side = "left"
tokenizer.padding_side = "right"

print("tokenizer.truncation_side =", tokenizer.truncation_side)


`torch_dtype` is deprecated! Use `dtype` instead!


Loaded: google/flan-t5-large | fp16: False | device: cpu
tokenizer.truncation_side = left


In [7]:
# ============================================================
# PROMPTS
# ============================================================

CODE_TO_LABEL = {"A": "support", "B": "deny", "C": "query", "D": "comment"}
LABEL_TO_CODE = {v: k for k, v in CODE_TO_LABEL.items()}
CODE_WORDS = ["A", "B", "C", "D"]

ZERO_SHOT_TEMPLATE = """You are an NLP classifier for rumour stance detection.

Choose exactly one code:
A = Support (agrees/affirms the rumour)
B = Deny (refutes/disagrees; calls it false)
C = Query (asks for evidence/clarification)
D = Comment (no clear stance; chatter/meta)

Decision rules:
- If the reply is a question / asks for proof -> C
- If it refutes, contradicts, says fake/false -> B
- If it confirms/affirms, says true -> A
- Use D ONLY if none of A/B/C apply. Do NOT default to D.

Return exactly one character: A, B, C, or D.

SOURCE: {source}
REPLY: {reply}

Answer:"""

FEW_SHOT_HEADER = """You are an NLP classifier for rumour stance detection.

Choose exactly one code:
A = Support
B = Deny
C = Query
D = Comment

Decision rules:
- Question / request for evidence -> C
- Refute / contradict / fake / false -> B
- Confirm / agree / true -> A
- D only if none of the above

Return exactly one character: A, B, C, or D.

Examples:
"""

FEW_SHOT_EXAMPLE_BLOCK = """Example {i}
SOURCE: {source}
REPLY: {reply}
Answer: {code}

"""

FEW_SHOT_QUERY_BLOCK = """Now classify the next pair.

SOURCE: {source}
REPLY: {reply}

Answer:"""


In [8]:


RE_QUERY = re.compile(r"\?$|\b(what|why|how|where|when|who)\b|\b(proof|evidence|source)\b", re.I)
RE_DENY  = re.compile(r"\b(fake|hoax|bullshit|bs|not true|no way|lies|lie|false|made up)\b", re.I)
RE_SUPP  = re.compile(r"\b(true|confirmed|real|legit|exactly|yes|yep|correct)\b", re.I)

def is_prototypical_row(gold: str, reply_text: str) -> bool:
    txt = str(reply_text)

    if gold == "query":
        return ("?" in txt) or bool(RE_QUERY.search(txt))
    if gold == "deny":
        return bool(RE_DENY.search(txt)) or ("not" in txt.lower())
    if gold == "support":
        return bool(RE_SUPP.search(txt))
    if gold == "comment":
        return ("?" not in txt) and (not RE_DENY.search(txt)) and (not RE_SUPP.search(txt))
    return True

def sample_few_shot_strong(df: pd.DataFrame, k_per_class: int = 4, seed: int = 42,
                          max_reply_chars: int = 240, max_source_chars: int = 240) -> pd.DataFrame:
    """
    Balanced, prototypical, short few-shot examples to reduce truncation.
    Returns only [SRC_COL, RPL_COL, gold].
    """
    rng = np.random.RandomState(seed)
    df = df.copy()

    df["_src_len"] = df[SRC_COL].astype(str).str.len()
    df["_rpl_len"] = df[RPL_COL].astype(str).str.len()

    df = df[(df["_src_len"] <= max_source_chars) & (df["_rpl_len"] <= max_reply_chars)].copy()

    df["_proto"] = df.apply(lambda r: is_prototypical_row(r["gold"], r[RPL_COL]), axis=1)

    shots = []
    for lab in ["deny", "query", "support", "comment"]:
        pool = df[df["gold"] == lab].copy()
        if len(pool) == 0:
            continue

        pool = pool.sort_values(by=["_proto", "_rpl_len", "_src_len"], ascending=[False, True, True])

        take = min(k_per_class, len(pool))
        topN = min(len(pool), max(40, take * 10))
        idx = rng.choice(pool.head(topN).index.values, size=take, replace=False)
        shots.append(pool.loc[idx])

    if not shots:
        return df.head(0)[[SRC_COL, RPL_COL, "gold"]].copy()

    out = pd.concat(shots).reset_index(drop=True)
    return out[[SRC_COL, RPL_COL, "gold"]].copy()

K_GRID = [1, 2, 4, 6]
print("K_GRID:", K_GRID)


K_GRID: [1, 2, 4, 6]


In [9]:
def _token_ids_for_code(code: str) -> List[int]:
    ids = tokenizer(code, add_special_tokens=False).input_ids
    return ids

CODE_TOKEN_IDS = {c: _token_ids_for_code(c) for c in CODE_WORDS}
print("Code token IDs:", CODE_TOKEN_IDS)

@torch.no_grad()
def score_single_token_codes(prompt: str) -> Tuple[str, Dict[str, float]]:
    if any(len(CODE_TOKEN_IDS[c]) != 1 for c in CODE_WORDS):
        return score_codes_fallback(prompt)

    enc = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(DEVICE)

    start_id = model.config.decoder_start_token_id
    if start_id is None:
        start_id = tokenizer.pad_token_id

    decoder_input_ids = torch.tensor([[start_id]], device=DEVICE)
    logits = model(**enc, decoder_input_ids=decoder_input_ids).logits
    logp = F.log_softmax(logits[0, 0], dim=-1)

    scores = {}
    for c in CODE_WORDS:
        tid = CODE_TOKEN_IDS[c][0]
        scores[c] = float(logp[tid].item())

    best_code = max(scores, key=scores.get)
    return CODE_TO_LABEL[best_code], scores

def score_target_sequence(prompt: str, target_ids: List[int]) -> float:
    enc = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(DEVICE)
    tgt = torch.tensor([target_ids], device=DEVICE)

    start_id = model.config.decoder_start_token_id
    if start_id is None:
        start_id = tokenizer.pad_token_id

    
    decoder_input_ids = torch.cat([torch.tensor([[start_id]], device=DEVICE), tgt[:, :-1]], dim=1)
    logits = model(**enc, decoder_input_ids=decoder_input_ids).logits
    logp = F.log_softmax(logits, dim=-1)

    
    total = 0.0
    for i in range(tgt.size(1)):
        total += float(logp[0, i, tgt[0, i]].item())
    return total

@torch.no_grad()
def score_codes_fallback(prompt: str) -> Tuple[str, Dict[str, float]]:
    scores = {}
    for c in CODE_WORDS:
        scores[c] = score_target_sequence(prompt, CODE_TOKEN_IDS[c])
    best_code = max(scores, key=scores.get)
    return CODE_TO_LABEL[best_code], scores


Code token IDs: {'A': [71], 'B': [272], 'C': [205], 'D': [309]}


In [10]:

def _clip_text(s: str, max_chars: int) -> str:
    s = str(s)
    s = re.sub(r"\s+", " ", s).strip()
    return s if len(s) <= max_chars else s[:max_chars-3] + "..."

def build_zero_shot_prompt(source: str, reply: str) -> str:
    return ZERO_SHOT_TEMPLATE.format(
        source=_clip_text(source, 300),
        reply=_clip_text(reply, 300)
    )

def build_few_shot_prompt(source: str, reply: str, few_shot_df: pd.DataFrame,
                          ex_src_chars: int = 140,
                          ex_rpl_chars: int = 160) -> str:
    """
    Few-shot prompt with strict per-example length caps to avoid truncation.
    """
    prompt = FEW_SHOT_HEADER

    
    for i, row in enumerate(few_shot_df.itertuples(index=False), start=1):
        gold = getattr(row, "gold")
        code = LABEL_TO_CODE[gold]
        prompt += FEW_SHOT_EXAMPLE_BLOCK.format(
            i=i,
            source=_clip_text(getattr(row, SRC_COL), ex_src_chars),
            reply=_clip_text(getattr(row, RPL_COL), ex_rpl_chars),
            code=code
        )

    
    prompt += FEW_SHOT_QUERY_BLOCK.format(
        source=_clip_text(source, 300),
        reply=_clip_text(reply, 300)
    )
    return prompt

def _token_ids_for_code(code: str) -> List[int]:
    return tokenizer(code, add_special_tokens=False).input_ids

CODE_TOKEN_IDS = {c: _token_ids_for_code(c) for c in CODE_WORDS}

@torch.no_grad()
def score_target_sequence(prompt: str, target_ids: List[int]) -> float:
    enc = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(DEVICE)
    tgt = torch.tensor([target_ids], device=DEVICE)

    start_id = model.config.decoder_start_token_id
    if start_id is None:
        start_id = tokenizer.pad_token_id

    decoder_input_ids = torch.cat([torch.tensor([[start_id]], device=DEVICE), tgt[:, :-1]], dim=1)
    logits = model(**enc, decoder_input_ids=decoder_input_ids).logits
    logp = F.log_softmax(logits, dim=-1)

    total = 0.0
    for i in range(tgt.size(1)):
        total += float(logp[0, i, tgt[0, i]].item())
    return total

@torch.no_grad()
def score_codes_fallback(prompt: str) -> Tuple[str, Dict[str, float]]:
    scores = {c: score_target_sequence(prompt, CODE_TOKEN_IDS[c]) for c in CODE_WORDS}
    best_code = max(scores, key=scores.get)
    return CODE_TO_LABEL[best_code], scores

@torch.no_grad()
def score_single_token_codes(prompt: str) -> Tuple[str, Dict[str, float]]:
    if any(len(CODE_TOKEN_IDS[c]) != 1 for c in CODE_WORDS):
        return score_codes_fallback(prompt)

    enc = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(DEVICE)

    start_id = model.config.decoder_start_token_id
    if start_id is None:
        start_id = tokenizer.pad_token_id

    decoder_input_ids = torch.tensor([[start_id]], device=DEVICE)
    logits = model(**enc, decoder_input_ids=decoder_input_ids).logits
    logp = F.log_softmax(logits[0, 0], dim=-1)

    scores = {}
    for c in CODE_WORDS:
        tid = CODE_TOKEN_IDS[c][0]
        scores[c] = float(logp[tid].item())

    best_code = max(scores, key=scores.get)
    return CODE_TO_LABEL[best_code], scores

def run_prompting_ranked(df: pd.DataFrame, mode: str, few_shot_df: pd.DataFrame = None,
                         limit: Optional[int] = None) -> pd.DataFrame:
    assert mode in ["zero", "few"]
    work = df.head(limit).copy() if limit else df.copy()

    rows = []
    for r in work.itertuples(index=False):
        source = getattr(r, SRC_COL)
        reply  = getattr(r, RPL_COL)
        gold   = getattr(r, "gold", None)

        if mode == "zero":
            prompt = build_zero_shot_prompt(source, reply)
        else:
            if few_shot_df is None:
                raise ValueError("few_shot_df must be provided for mode='few'")
            prompt = build_few_shot_prompt(source, reply, few_shot_df)

        pred, scores = score_single_token_codes(prompt)

        sorted_scores = sorted(scores.items(), key=lambda x: x[1], reverse=True)
        margin = float(sorted_scores[0][1] - sorted_scores[1][1])

        rows.append({
            "gold": gold,
            "pred": pred,
            "margin": margin,
            "scores": scores,
            "source": source,
            "reply": reply,
        })

    return pd.DataFrame(rows)

print("Core functions updated. Left-truncation compatible + few-shot length control")


Core functions updated. Left-truncation compatible + few-shot length control


In [11]:
TUNE_DEV_LIMIT = 400 


dev_zero_tune = run_prompting_ranked(dev_df, mode="zero", limit=TUNE_DEV_LIMIT)
dev_zero_macro_tune = float(f1_score(dev_zero_tune["gold"], dev_zero_tune["pred"],
                                     average="macro", labels=LABELS_CANON))
print("DEV zero-shot macro-F1 (tune subset):", round(dev_zero_macro_tune, 4))

best_k = None
best_m = None
best_shots = None

for k in K_GRID:
    shots = sample_few_shot_strong(train_df, k_per_class=k, seed=SEED)
    dev_few_tune = run_prompting_ranked(dev_df, mode="few", few_shot_df=shots, limit=TUNE_DEV_LIMIT)

    m = float(f1_score(dev_few_tune["gold"], dev_few_tune["pred"],
                       average="macro", labels=LABELS_CANON))
    print(f"k_per_class={k} -> DEV few-shot macro-F1 (tune subset) = {m:.4f}")

    if best_m is None or m > best_m:
        best_m = m
        best_k = k
        best_shots = shots

print("\nSelected k_per_class =", best_k, "tune macro-F1 =", round(best_m, 4))

few_shot_df = best_shots.copy()
display(few_shot_df)


DEV zero-shot macro-F1 (tune subset): 0.4278
k_per_class=1 -> DEV few-shot macro-F1 (tune subset) = 0.4028
k_per_class=2 -> DEV few-shot macro-F1 (tune subset) = 0.0734
k_per_class=4 -> DEV few-shot macro-F1 (tune subset) = 0.0726
k_per_class=6 -> DEV few-shot macro-F1 (tune subset) = 0.0726

Selected k_per_class = 1 tune macro-F1 = 0.4028


,source_text__unified,reply_text__unified,gold
0,BREAKING NEWS: New York Times is reporting the...,<USER> Not according to what I've just heard...,deny
1,#BREAKING Rick Hughes at CBC Hamilton: \n\nThe...,<USER> <USER> is he alright?,query
2,If we shot everyone in the street for stealing...,<USER> <USER> yes but cops only shoot for ...,support
3,Hostages in Sydney cafe made to hold up Islami...,<USER> Pigs,comment


In [12]:


MAX_TOKENS = 512
DIAG_LIMIT = 500

def prompt_token_count(prompt: str) -> int:
    return len(tokenizer(prompt, truncation=False).input_ids)

def diagnose_prompt_lengths(df: pd.DataFrame, split_name: str, limit: int = None):
    work = df.head(limit).copy() if limit else df.copy()

    zero_over = 0
    few_over = 0
    total = len(work)

    zero_lens = []
    few_lens = []

    for r in work.itertuples(index=False):
        src = getattr(r, SRC_COL)
        rpl = getattr(r, RPL_COL)

        p_zero = build_zero_shot_prompt(src, rpl)
        p_few  = build_few_shot_prompt(src, rpl, few_shot_df)

        z_len = prompt_token_count(p_zero)
        f_len = prompt_token_count(p_few)

        zero_lens.append(z_len)
        few_lens.append(f_len)

        if z_len > MAX_TOKENS:
            zero_over += 1
        if f_len > MAX_TOKENS:
            few_over += 1

    print("\n" + "="*70)
    print(f"PROMPT LENGTH DIAGNOSTIC — {split_name.upper()}")
    print(f"Samples checked: {total}")
    print(f"ZERO-SHOT: {zero_over}/{total} over limit "
          f"({100*zero_over/total:.1f}%) | mean={int(np.mean(zero_lens))} | max={max(zero_lens)}")
    print(f"FEW-SHOT : {few_over}/{total} over limit "
          f"({100*few_over/total:.1f}%) | mean={int(np.mean(few_lens))} | max={max(few_lens)}")

    return {
        "zero_over_pct": 100 * zero_over / total,
        "few_over_pct": 100 * few_over / total,
        "zero_mean": float(np.mean(zero_lens)),
        "few_mean": float(np.mean(few_lens)),
        "zero_max": int(max(zero_lens)),
        "few_max": int(max(few_lens)),
    }


diag_dev  = diagnose_prompt_lengths(dev_df,  "dev",  limit=DIAG_LIMIT)
diag_test = diagnose_prompt_lengths(test_df, "test", limit=DIAG_LIMIT)



PROMPT LENGTH DIAGNOSTIC — DEV
Samples checked: 500
ZERO-SHOT: 0/500 over limit (0.0%) | mean=252 | max=299
FEW-SHOT : 0/500 over limit (0.0%) | mean=411 | max=458

PROMPT LENGTH DIAGNOSTIC — TEST
Samples checked: 500
ZERO-SHOT: 0/500 over limit (0.0%) | mean=199 | max=199
FEW-SHOT : 0/500 over limit (0.0%) | mean=358 | max=358


In [13]:
def pretty_report(df_pred: pd.DataFrame, name: str):
    df_pred = df_pred.dropna(subset=["gold"]).copy()
    y_true = df_pred["gold"].tolist()
    y_pred = df_pred["pred"].tolist()

    macro = f1_score(y_true, y_pred, average="macro", labels=LABELS_CANON)
    print("\n" + "="*70)
    print(name, "| n =", len(df_pred), "| macro-F1 =", round(macro, 4))

    print("\nLabel counts (gold):", pd.Series(y_true).value_counts().to_dict())
    print("Label counts (pred):", pd.Series(y_pred).value_counts().to_dict())

    print("\nClassification report:")
    print(classification_report(y_true, y_pred, labels=LABELS_CANON, digits=3, zero_division=0))

    cm = confusion_matrix(y_true, y_pred, labels=LABELS_CANON)
    cm_df = pd.DataFrame(cm, index=[f"gold={l}" for l in LABELS_CANON],
                         columns=[f"pred={l}" for l in LABELS_CANON])
    print("\nConfusion matrix:")
    display(cm_df)

    return macro, cm_df


In [14]:

DEV_LIMIT = None

t0 = time.time()
dev_zero = run_prompting_ranked(dev_df, mode="zero", limit=DEV_LIMIT)
print("Dev ZERO done in", round(time.time() - t0, 2), "s")

t0 = time.time()
dev_few = run_prompting_ranked(dev_df, mode="few", few_shot_df=few_shot_df, limit=DEV_LIMIT)
print("Dev FEW done in", round(time.time() - t0, 2), "s")

dev_zero_macro, _ = pretty_report(dev_zero, "DEV ZERO-SHOT (ranked)")
dev_few_macro,  _ = pretty_report(dev_few,  f"DEV FEW-SHOT (ranked) | k_per_class={best_k}")


BEST_MODE = "few" if dev_few_macro >= dev_zero_macro else "zero"
print("\nRECOMMENDATION (based on DEV):", BEST_MODE.upper())


Dev ZERO done in 508.06 s
Dev FEW done in 698.48 s

DEV ZERO-SHOT (ranked) | n = 950 | macro-F1 = 0.4484

Label counts (gold): {'comment': 616, 'support': 167, 'query': 96, 'deny': 71}
Label counts (pred): {'comment': 581, 'support': 183, 'query': 165, 'deny': 21}

Classification report:
              precision    recall  f1-score   support

     support      0.497     0.545     0.520       167
        deny      0.143     0.042     0.065        71
       query      0.364     0.625     0.460        96
     comment      0.771     0.727     0.749       616

    accuracy                          0.634       950
   macro avg      0.444     0.485     0.448       950
weighted avg      0.635     0.634     0.628       950


Confusion matrix:


,pred=support,pred=deny,pred=query,pred=comment
gold=support,91,3,14,59
gold=deny,6,3,19,43
gold=query,3,2,60,31
gold=comment,83,13,72,448



DEV FEW-SHOT (ranked) | k_per_class=1 | n = 950 | macro-F1 = 0.4144

Label counts (gold): {'comment': 616, 'support': 167, 'query': 96, 'deny': 71}
Label counts (pred): {'comment': 659, 'query': 153, 'support': 102, 'deny': 36}

Classification report:
              precision    recall  f1-score   support

     support      0.608     0.371     0.461       167
        deny      0.056     0.028     0.037        71
       query      0.333     0.531     0.410        96
     comment      0.725     0.776     0.750       616

    accuracy                          0.624       950
   macro avg      0.431     0.427     0.414       950
weighted avg      0.615     0.624     0.611       950


Confusion matrix:


,pred=support,pred=deny,pred=query,pred=comment
gold=support,62,5,17,83
gold=deny,3,2,8,58
gold=query,1,4,51,40
gold=comment,36,25,77,478



RECOMMENDATION (based on DEV): ZERO


In [15]:

from itertools import product
from sklearn.metrics import f1_score, classification_report

def apply_offsets_to_scores(scores_dict, offsets_label: dict):
    adj = {}
    for code, base in scores_dict.items():
        lab = CODE_TO_LABEL[code]
        adj[code] = float(base) + float(offsets_label.get(lab, 0.0))
    return adj

def predict_with_offsets(df_pred, offsets_label: dict, pred_col: str = "pred_cal"):
    out = df_pred.copy()
    preds = []
    for s in out["scores"]:
        adj = apply_offsets_to_scores(s, offsets_label)
        best_code = max(adj, key=adj.get)
        preds.append(CODE_TO_LABEL[best_code])
    out[pred_col] = preds
    return out

def macro_f1(df_pred, pred_col: str):
    return float(f1_score(df_pred["gold"], df_pred[pred_col], average="macro", labels=LABELS_CANON))

def grid_search_offsets(df_pred, 
                        comment_offsets=(-3.0,-2.0,-1.5,-1.0,-0.75,-0.5,-0.25,0.0),
                        sdq_boosts=(0.0,0.25,0.5,0.75,1.0,1.5,2.0)):
    best = None
    best_df = None

    for com_off, sdq in product(comment_offsets, sdq_boosts):
        offsets = {
            "comment": com_off,
            "support": sdq,
            "deny": sdq,
            "query": sdq,
        }
        tmp = predict_with_offsets(df_pred, offsets, pred_col="pred_cal")
        m = macro_f1(tmp, "pred_cal")

        if best is None or m > best["macro_f1"]:
            best = {"macro_f1": m, "offsets": offsets}
            best_df = tmp

    return best, best_df

def print_calibration_report(name: str, df_base, df_cal):
    base_m = float(f1_score(df_base["gold"], df_base["pred"], average="macro", labels=LABELS_CANON))
    cal_m  = float(f1_score(df_cal["gold"],  df_cal["pred_cal"], average="macro", labels=LABELS_CANON))
    print("\n" + "="*70)
    print(name)
    print("Base macro-F1:", round(base_m, 4))
    print("Cal  macro-F1:", round(cal_m, 4))
    print("Δ:", round(cal_m - base_m, 4))
    print("\nCalibrated classification report:")
    print(classification_report(df_cal["gold"], df_cal["pred_cal"], labels=LABELS_CANON, digits=3, zero_division=0))


In [16]:

assert "scores" in dev_zero.columns, "dev_zero missing 'scores' dict column"
assert "scores" in dev_few.columns,  "dev_few missing 'scores' dict column"

best_zero, dev_zero_cal = grid_search_offsets(dev_zero)
best_few,  dev_few_cal  = grid_search_offsets(dev_few)

CAL_OFFSETS_ZERO = best_zero["offsets"]
CAL_OFFSETS_FEW  = best_few["offsets"]

print("Best DEV calibration (ZERO):", round(best_zero["macro_f1"], 4), "offsets=", CAL_OFFSETS_ZERO)
print("Best DEV calibration (FEW) :", round(best_few["macro_f1"], 4),  "offsets=", CAL_OFFSETS_FEW)

print_calibration_report("DEV ZERO-SHOT CALIBRATION", dev_zero, dev_zero_cal)
print_calibration_report("DEV FEW-SHOT  CALIBRATION", dev_few,  dev_few_cal)

dev_zero_macro_cal = macro_f1(dev_zero_cal, "pred_cal")
dev_few_macro_cal  = macro_f1(dev_few_cal,  "pred_cal")

BEST_MODE_CAL = "few" if dev_few_macro_cal >= dev_zero_macro_cal else "zero"


Best DEV calibration (ZERO): 0.4484 offsets= {'comment': 0.0, 'support': 0.0, 'deny': 0.0, 'query': 0.0}
Best DEV calibration (FEW) : 0.4168 offsets= {'comment': -0.25, 'support': 0.0, 'deny': 0.0, 'query': 0.0}

DEV ZERO-SHOT CALIBRATION
Base macro-F1: 0.4484
Cal  macro-F1: 0.4484
Δ: 0.0

Calibrated classification report:
              precision    recall  f1-score   support

     support      0.497     0.545     0.520       167
        deny      0.143     0.042     0.065        71
       query      0.364     0.625     0.460        96
     comment      0.771     0.727     0.749       616

    accuracy                          0.634       950
   macro avg      0.444     0.485     0.448       950
weighted avg      0.635     0.634     0.628       950


DEV FEW-SHOT  CALIBRATION
Base macro-F1: 0.4144
Cal  macro-F1: 0.4168
Δ: 0.0023

Calibrated classification report:
              precision    recall  f1-score   support

     support      0.553     0.407     0.469       167
        deny   

In [17]:

TEST_LIMIT = None


test_pred_zero = run_prompting_ranked(test_df, mode="zero", limit=TEST_LIMIT)
test_pred_few  = run_prompting_ranked(test_df, mode="few", few_shot_df=few_shot_df, limit=TEST_LIMIT)


test_zero_base = float(f1_score(test_pred_zero["gold"], test_pred_zero["pred"], average="macro", labels=LABELS_CANON))
test_few_base  = float(f1_score(test_pred_few["gold"],  test_pred_few["pred"],  average="macro", labels=LABELS_CANON))


test_zero_cal_df = predict_with_offsets(test_pred_zero, CAL_OFFSETS_ZERO, pred_col="pred_cal")
test_few_cal_df  = predict_with_offsets(test_pred_few,  CAL_OFFSETS_FEW,  pred_col="pred_cal")

test_zero_cal = float(f1_score(test_zero_cal_df["gold"], test_zero_cal_df["pred_cal"], average="macro", labels=LABELS_CANON))
test_few_cal  = float(f1_score(test_few_cal_df["gold"],  test_few_cal_df["pred_cal"],  average="macro", labels=LABELS_CANON))

print("\n=== TEST RESULTS (BASE vs CALIBRATED) ===")
print("TEST ZERO base macro-F1:", round(test_zero_base, 4))
print("TEST ZERO cal  macro-F1:", round(test_zero_cal, 4))
print("TEST FEW  base macro-F1:", round(test_few_base, 4))
print("TEST FEW  cal  macro-F1:", round(test_few_cal, 4))


macro_test_zero_base = test_zero_base
macro_test_few_base  = test_few_base
macro_test_zero_cal  = test_zero_cal
macro_test_few_cal   = test_few_cal


print("\nRecommended mode (calibrated DEV):", BEST_MODE_CAL.upper())


test_pred_zero_cal = test_zero_cal_df
test_pred_few_cal  = test_few_cal_df



=== TEST RESULTS (BASE vs CALIBRATED) ===
TEST ZERO base macro-F1: 0.2129
TEST ZERO cal  macro-F1: 0.2129
TEST FEW  base macro-F1: 0.2129
TEST FEW  cal  macro-F1: 0.2129

Recommended mode (calibrated DEV): ZERO


In [18]:


def failure_summary(df_pred: pd.DataFrame, name: str):
    dfp = df_pred.copy()
    default_comment_rate = float((dfp["pred"] == "comment").mean())
    minority_to_comment = dfp[(dfp["gold"].isin(["support","deny","query"])) & (dfp["pred"] == "comment")]

    print("\n" + "="*70)
    print(name)
    print("Default-to-comment rate:", round(default_comment_rate, 4))
    print("Minority→Comment errors:", len(minority_to_comment))


    uncertain = dfp.sort_values("margin", ascending=True).head(12)
    print("\nMost uncertain (lowest margin):")
    display(uncertain[["gold","pred","margin","source","reply"]])


    print("\nMinority→Comment examples:")
    display(minority_to_comment.head(12)[["gold","pred","margin","source","reply"]])

    return default_comment_rate, minority_to_comment

dc0, m2c0 = failure_summary(test_pred_zero, "FAILURE MODES — TEST ZERO-SHOT")
dcf, m2cf = failure_summary(test_pred_few,  "FAILURE MODES — TEST FEW-SHOT")



FAILURE MODES — TEST ZERO-SHOT
Default-to-comment rate: 1.0
Minority→Comment errors: 271

Most uncertain (lowest margin):


,gold,pred,margin,source,reply
0,comment,comment,0.072035,nan,nan
690,support,comment,0.072035,nan,nan
691,comment,comment,0.072035,nan,nan
692,comment,comment,0.072035,nan,nan
693,comment,comment,0.072035,nan,nan
694,comment,comment,0.072035,nan,nan
695,comment,comment,0.072035,nan,nan
696,comment,comment,0.072035,nan,nan
697,comment,comment,0.072035,nan,nan
698,comment,comment,0.072035,nan,nan



Minority→Comment examples:


,gold,pred,margin,source,reply
3,query,comment,0.072035,nan,nan
4,deny,comment,0.072035,nan,nan
11,query,comment,0.072035,nan,nan
15,query,comment,0.072035,nan,nan
19,query,comment,0.072035,nan,nan
28,support,comment,0.072035,nan,nan
30,query,comment,0.072035,nan,nan
35,query,comment,0.072035,nan,nan
40,query,comment,0.072035,nan,nan
42,query,comment,0.072035,nan,nan



FAILURE MODES — TEST FEW-SHOT
Default-to-comment rate: 1.0
Minority→Comment errors: 271

Most uncertain (lowest margin):


,gold,pred,margin,source,reply
0,comment,comment,1.069716,nan,nan
690,support,comment,1.069716,nan,nan
691,comment,comment,1.069716,nan,nan
692,comment,comment,1.069716,nan,nan
693,comment,comment,1.069716,nan,nan
694,comment,comment,1.069716,nan,nan
695,comment,comment,1.069716,nan,nan
696,comment,comment,1.069716,nan,nan
697,comment,comment,1.069716,nan,nan
698,comment,comment,1.069716,nan,nan



Minority→Comment examples:


,gold,pred,margin,source,reply
3,query,comment,1.069716,nan,nan
4,deny,comment,1.069716,nan,nan
11,query,comment,1.069716,nan,nan
15,query,comment,1.069716,nan,nan
19,query,comment,1.069716,nan,nan
28,support,comment,1.069716,nan,nan
30,query,comment,1.069716,nan,nan
35,query,comment,1.069716,nan,nan
40,query,comment,1.069716,nan,nan
42,query,comment,1.069716,nan,nan


In [19]:
print("\n=== DEV (CALIBRATED) ===")
print("DEV zero cal macro-F1:", round(dev_zero_macro_cal, 4))
print("DEV few  cal macro-F1:", round(dev_few_macro_cal, 4))
print("BEST_MODE_CAL:", BEST_MODE_CAL.upper())

print("\n=== TEST (BASE vs CALIBRATED) ===")
print("ZERO base:", round(macro_test_zero_base, 4), "| cal:", round(macro_test_zero_cal, 4))
print("FEW  base:", round(macro_test_few_base, 4),  "| cal:", round(macro_test_few_cal, 4))



=== DEV (CALIBRATED) ===
DEV zero cal macro-F1: 0.4484
DEV few  cal macro-F1: 0.4168
BEST_MODE_CAL: ZERO

=== TEST (BASE vs CALIBRATED) ===
ZERO base: 0.2129 | cal: 0.2129
FEW  base: 0.2129 | cal: 0.2129
